# 🚗 Autonomous Vehicle — Behavioral Cloning (Google Colab)

**Run this notebook in Google Colab for free GPU training.**

**Workflow:**
1. Run all cells here on Colab (trains the model)
2. Download `best_model.h5` from Google Drive
3. Run `drive.py` locally with the Udacity Simulator

---
**Days covered:** Day 2 (Dataset), Day 3 (Exploration), Day 4 (Augmentation), Day 5-6 (Model + Training)

## ✅ Step 0 — Check GPU
Make sure Colab is using GPU: Runtime → Change runtime type → GPU

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
# If GPU list is empty → go to Runtime > Change runtime type > GPU

## 📥 Day 2 — Download & Extract the Udacity Dataset

In [ ]:
import os, zipfile, urllib.request

DATA_URL  = 'https://d17h27t6h515a5.cloudfront.net/topher/2016/December/584f6edd_data/data.zip'
ZIP_PATH  = '/content/data.zip'
DATA_PATH = '/content/data/'

if not os.path.exists(DATA_PATH):
    print('Downloading dataset (~300MB)... please wait')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Download complete. Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done! Dataset ready at:', DATA_PATH)
else:
    print('Dataset already exists at:', DATA_PATH)

import glob
imgs = glob.glob(DATA_PATH + 'IMG/*.jpg')
print(f'Total images found: {len(imgs)}')

## 🔍 Day 3 — Load & Explore the Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import random
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
print('Libraries imported.')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
LOG_FILE      = DATA_PATH + 'driving_log.csv'
IMG_HEIGHT    = 66
IMG_WIDTH     = 200
IMG_CHANNELS  = 3
CORRECTION    = 0.2    # steering correction for left/right cameras
BATCH_SIZE    = 32
EPOCHS        = 15
LEARNING_RATE = 1e-4
# ───────────────────────────────────────────────────────────────────────────

cols = ['center','left','right','steering','throttle','brake','speed']
df = pd.read_csv(LOG_FILE, names=cols, header=None)
print(f'Total rows in driving log: {len(df)}')
df.head()

In [ ]:
# ── Steering angle statistics ──────────────────────────────────────────────
print('Steering Angle Statistics:')
print(df['steering'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['steering'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Steering Angle Distribution (Raw)')
axes[0].set_xlabel('Steering Angle')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['speed'], bins=30, color='coral', edgecolor='black')
axes[1].set_title('Speed Distribution')
axes[1].set_xlabel('Speed (mph)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
print('Observation: Most steering angles are near 0 (straight driving). Augmentation will balance this.')

In [ ]:
# ── Show sample images from all 3 cameras ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, cam in enumerate(['center', 'left', 'right']):
    fname = df[cam].iloc[10].strip().split('/')[-1]
    img = mpimg.imread(DATA_PATH + 'IMG/' + fname)
    axes[i].imshow(img)
    axes[i].set_title(f'{cam.capitalize()} Camera')
    axes[i].axis('off')
plt.suptitle('Sample Images from Dataset (same moment, 3 cameras)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Build sample list using all 3 cameras ─────────────────────────────────
samples = []
for _, row in df.iterrows():
    sc = float(row['steering'])
    for cam, angle in [('center', sc),
                        ('left',   sc + CORRECTION),
                        ('right',  sc - CORRECTION)]:
        fname = row[cam].strip().split('/')[-1]
        path  = DATA_PATH + 'IMG/' + fname
        samples.append((path, angle))

train_samples, val_samples = train_test_split(samples, test_size=0.2, random_state=42)
print(f'Total samples (3 cameras): {len(samples)}')
print(f'Training  : {len(train_samples)}')
print(f'Validation: {len(val_samples)}')

## 🔄 Day 4 — Data Augmentation
We apply 3 augmentations to combat overfitting and handle imbalanced data:
- **Random Flip** — Mirror image, negate angle
- **Random Pan** — Shift image horizontally/vertically
- **Random Brightness** — Simulate day/night/shadow

In [ ]:
def preprocess(img):
    """Crop sky/hood, resize to 66x200, convert to YUV."""
    img = img[60:-25, :, :]                        # crop
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))  # resize
    img = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)      # YUV
    return img

def random_flip(img, angle):
    if random.random() > 0.5:
        img   = cv2.flip(img, 1)
        angle = -angle
    return img, angle

def random_pan(img, angle):
    px  = 100 * (random.random() - 0.5)
    py  = 10  * (random.random() - 0.5)
    M   = np.float32([[1, 0, px], [0, 1, py]])
    h, w = img.shape[:2]
    img  = cv2.warpAffine(img, M, (w, h))
    angle += px * 0.002
    return img, angle

def random_brightness(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    f   = 0.25 + random.random()
    hsv[:,:,2] = np.clip(hsv[:,:,2].astype(np.float32) * f, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def augment(img, angle):
    img, angle = random_pan(img, angle)
    img        = random_brightness(img)
    img, angle = random_flip(img, angle)
    return img, angle

print('Augmentation functions ready.')

In [ ]:
# ── Visualize augmentations ────────────────────────────────────────────────
path, angle = train_samples[5]
orig = cv2.imread(path)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

# Row 1: individual augmentations
axes[0][0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
axes[0][0].set_title(f'Original\nAngle: {angle:.3f}')

flipped, fa = random_flip(orig.copy(), angle)
axes[0][1].imshow(cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB))
axes[0][1].set_title(f'After Flip\nAngle: {fa:.3f}')

panned, pa = random_pan(orig.copy(), angle)
axes[0][2].imshow(cv2.cvtColor(panned, cv2.COLOR_BGR2RGB))
axes[0][2].set_title(f'After Pan\nAngle: {pa:.3f}')

brightened = random_brightness(orig.copy())
axes[0][3].imshow(cv2.cvtColor(brightened, cv2.COLOR_BGR2RGB))
axes[0][3].set_title('After Brightness')

# Row 2: full augment pipeline + preprocessed
for i in range(3):
    aug, aa = augment(orig.copy(), angle)
    axes[1][i].imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
    axes[1][i].set_title(f'Full Augment #{i+1}\nAngle: {aa:.3f}')

pre = preprocess(orig.copy())
axes[1][3].imshow(pre)  # YUV display
axes[1][3].set_title(f'Preprocessed (YUV)\n{IMG_HEIGHT}x{IMG_WIDTH}')

for ax in axes.flatten(): ax.axis('off')
plt.suptitle('Data Augmentation Pipeline', fontsize=15)
plt.tight_layout()
plt.show()

## ⚙️ Data Generator

In [ ]:
def data_generator(samples, batch_size=32, is_training=True):
    while True:
        samples = shuffle(samples)
        for offset in range(0, len(samples), batch_size):
            batch  = samples[offset:offset+batch_size]
            images, angles = [], []
            for p, a in batch:
                img = cv2.imread(p)
                if img is None: continue
                if is_training:
                    img, a = augment(img, a)
                img = preprocess(img).astype(np.float32) / 255.0
                images.append(img)
                angles.append(a)
            yield np.array(images), np.array(angles)

train_gen = data_generator(train_samples, BATCH_SIZE, is_training=True)
val_gen   = data_generator(val_samples,   BATCH_SIZE, is_training=False)
print('Generators ready.')

## 🧠 Day 5 — Build NVIDIA CNN (from scratch)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

def build_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    """
    NVIDIA End-to-End CNN for self-driving (built from scratch).
    Reference: Bojarski et al., 2016 — arxiv.org/abs/1604.07316
    """
    model = Sequential([
        # 5 Convolutional layers — extract road features
        Conv2D(24, (5,5), strides=(2,2), activation='elu', input_shape=input_shape, name='conv1'),
        Conv2D(36, (5,5), strides=(2,2), activation='elu', name='conv2'),
        Conv2D(48, (5,5), strides=(2,2), activation='elu', name='conv3'),
        Conv2D(64, (3,3), activation='elu', name='conv4'),
        Conv2D(64, (3,3), activation='elu', name='conv5'),
        Dropout(0.5),          # prevent overfitting
        Flatten(),
        # 4 Fully connected layers — decide steering angle
        Dense(100, activation='elu', name='fc1'),
        Dense(50,  activation='elu', name='fc2'),
        Dense(10,  activation='elu', name='fc3'),
        Dense(1,   name='output')   # single steering angle output
    ])
    return model

model = build_model()
model.summary()

## 🏋️ Day 6 — Train the Model

In [ ]:
# ── Mount Google Drive to save model ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
MODEL_SAVE_PATH = '/content/drive/MyDrive/autonomous_vehicle/best_model.h5'
import os; os.makedirs('/content/drive/MyDrive/autonomous_vehicle/', exist_ok=True)
print('Google Drive mounted. Model will save to:', MODEL_SAVE_PATH)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
]

steps   = len(train_samples) // BATCH_SIZE
v_steps = len(val_samples)   // BATCH_SIZE

print(f'Training: {steps} steps/epoch | Validation: {v_steps} steps')

history = model.fit(
    train_gen,
    steps_per_epoch=steps,
    validation_data=val_gen,
    validation_steps=v_steps,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
print('Training complete! Model saved to Google Drive.')

## 📊 Day 7 — Evaluate Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='coral')
axes[0].set_title('Loss (MSE) Over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='Train MAE', color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val MAE',   color='coral')
axes[1].set_title('Mean Absolute Error Over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (radians)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/autonomous_vehicle/training_results.png', dpi=120)
plt.show()
print('Chart saved to Google Drive.')

In [ ]:
# ── Predictions on 6 random validation images ─────────────────────────────
from tensorflow.keras.models import load_model
best = load_model(MODEL_SAVE_PATH)

test_s = random.sample(val_samples, 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (p, true_a) in enumerate(test_s):
    img = cv2.imread(p)
    pre = preprocess(img).astype(np.float32) / 255.0
    pred_a = best.predict(np.expand_dims(pre, 0), verbose=0)[0][0]
    err  = abs(pred_a - true_a)
    col  = 'green' if err < 0.05 else 'orange' if err < 0.15 else 'red'
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'True: {true_a:.3f} | Pred: {pred_a:.3f}\nErr: {err:.3f}', color=col)
    axes[i].axis('off')

plt.suptitle('Steering Angle Predictions vs Ground Truth', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/autonomous_vehicle/predictions.png', dpi=120)
plt.show()

## 🚗 Day 8 — Run Locally with the Simulator

After training is done:

1. **Download** `best_model.h5` from Google Drive → put it in your project folder
2. **Install dependencies locally:**
```bash
pip install -r requirements.txt
```
3. **Open** `Default Windows desktop 64-bit.exe` (the Udacity simulator)
4. Choose a track → click **Autonomous Mode**
5. **Run the server:**
```bash
python drive.py best_model.h5
```
6. 🎉 Watch the car drive itself!